<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 6th exercise: <font color="#C70039">Perform data augmentation for the sample solution in exercise 6</font>
* Course: AML
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date: 10.08.2026

<img src="http://ai.stanford.edu/blog/assets/img/posts/2020-04-20-data-augmentation/thumbnail.png" style="float: center;" width="450">

---------------------------------
**GENERAL NOTE 1**:  
Please read the entire notebook. Its markdown cells and code comments explain how the individual steps work together.

---------------------

### <font color="ce33ff">DESCRIPTION</font>:
This notebook uses PyTorch and torchvision to augment images and save the generated variants to disk.

-------------------------------------------------------------------------------------------------------------

In [ ]:
# Google Colab setup: make repository files available under the expected relative paths.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repository = "/content/AML"
    if not os.path.isdir(repository):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gheisenberg/AML.git", repository], check=True)
    os.chdir(repository)

In [ ]:
# torchvision v2 supplies modern, PyTorch-compatible image transformations.
# Path objects make the input and output folders explicit and portable.
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms import v2

torch.manual_seed(1)

In [ ]:
# Each pipeline call creates a different random image variant.
# Change PATH_TO_IMAGES before running this cell.
augmentation = v2.Compose(
    [
        v2.RandomAffine(degrees=90, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
    ]
)

# Change this to a folder of your own images when experimenting.
image_folder = Path("./data/einstein_mona_lisa/einstein")
if not image_folder.is_dir():
    raise FileNotFoundError(f"Image folder does not exist: {image_folder}")

output_folder = image_folder / "augmented"
output_folder.mkdir(exist_ok=True)

supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_paths = sorted(
    path for path in image_folder.iterdir()
    if path.is_file() and path.suffix.lower() in supported_extensions
)
print("Images:", [path.name for path in image_paths])

num_augmented_images = 50
for image_path in image_paths:
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        for index in range(num_augmented_images):
            augmented_image = augmentation(image)
            output_path = output_folder / f"{image_path.stem}_aug_{index:03d}.jpg"
            augmented_image.save(output_path, quality=95)